# 🚀 SETUP PARA GOOGLE COLAB
Ejecuta esta celda PRIMERO para instalar las dependencias y descargar los datos

In [ ]:
# Instalar dependencias
!pip install pandas numpy matplotlib seaborn plotly scikit-learn scipy openpyxl -q

# Clonar el repositorio
!git clone https://github.com/Jextends212/Reto-Aux-Analista.git /content/proyecto -q

# Cambiar a la carpeta del proyecto
import os
os.chdir('/content/proyecto')

print("✅ Setup completado. Los datos están listos.")
print("Archivos disponibles:")
import subprocess
subprocess.run(['ls', '-la', 'data/'], check=False)

# Análisis Exploratorio de Datos - Reto Alkomprar

**Autor: Juan Esteban Henao Palacio**  
**Fecha: 26 Abril del 2026**  
**Plataforma: Google Colab**  

---

## Importar Librerías

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
print("✅ Todas las librerías importadas correctamente")

## Cargar Datos

In [ ]:
excel_path = 'data/Prueba_Tecnica_Base_BI.xlsx'

# Verificar que el archivo existe
if os.path.exists(excel_path):
    print(f"✅ Archivo encontrado: {excel_path}")
else:
    print(f"❌ Archivo NO encontrado: {excel_path}")

excel_file = pd.ExcelFile(excel_path)

# Leer las 3 hojas
df_orders = pd.read_excel(excel_path, sheet_name='Ordenes')
df_products = pd.read_excel(excel_path, sheet_name='Producto')
df_regions = pd.read_excel(excel_path, sheet_name='Region')

print(f"\n✅ Información de los dataframes:")
print(f"  - Órdenes: {df_orders.shape[0]:,} filas × {df_orders.shape[1]} columnas")
print(f"  - Productos: {df_products.shape[0]:,} filas × {df_products.shape[1]} columnas")
print(f"  - Regiones: {df_regions.shape[0]:,} filas × {df_regions.shape[1]} columnas")

## Información detallada de Órdenes

In [ ]:
print("Información hoja de Órdenes:")
print(f"\nMemoria usada: {df_orders.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nPrimeras 5 filas:")
df_orders.head()

## Exploración de Datos

In [ ]:
print("\n=" * 80)
print("RESUMEN EJECUTIVO DE DATOS")
print("=" * 80)

# Preguntas Clave
total_ordenes = df_orders["ID_orden"].nunique()
fecha_min = df_orders["Fecha_de_la_orden"].min()
fecha_max = df_orders["Fecha_de_la_orden"].max()
periodo_dias = (fecha_max - fecha_min).days

print(f"\n📊 ÓRDENES Y PERÍODO:")
print(f"   Total de órdenes únicas: {total_ordenes:,}")
print(f"   Período: {fecha_min.strftime('%Y-%m-%d')} a {fecha_max.strftime('%Y-%m-%d')}")
print(f"   Duración: {periodo_dias} días (~{periodo_dias//365} años)")

# Clientes únicos
clientes_unicos = df_orders["Nombre_del_cliente"].nunique()
print(f"\n👥 CLIENTES:")
print(f"   Total de clientes únicos: {clientes_unicos:,}")
print(f"   Promedio de órdenes por cliente: {total_ordenes / clientes_unicos:.1f}")

# Monto promedio
total_ventas = df_orders["Ventas"].sum()
monto_promedio_orden = df_orders.groupby("ID_orden")["Ventas"].sum().mean()

print(f"\n💰 MONTO DE ORDEN:")
print(f"   Total de ventas: ${total_ventas:,.2f}")
print(f"   Monto promedio por orden: ${monto_promedio_orden:,.2f}")

# Productos
productos_unicos = df_orders["ID_producto"].nunique()
print(f"\n📦 PRODUCTOS:")
print(f"   Total de productos únicos: {productos_unicos:,}")
print(f"   Categorías únicas: {df_orders['Categoría'].nunique()}")

# Top países
print(f"\n🌍 TOP 5 PAÍSES (por ventas):")
top_paises = df_orders.groupby("País")["Ventas"].agg(["sum", "count"]).sort_values("sum", ascending=False)
top_paises.columns = ["Ventas_Total", "Num_Ordenes"]
for idx, (pais, row) in enumerate(top_paises.head(5).iterrows(), 1):
    pct_ventas = (row["Ventas_Total"] / total_ventas) * 100
    print(f"   {idx}. {pais}: ${row['Ventas_Total']:,.0f} ({pct_ventas:.1f}%) - {int(row['Num_Ordenes'])} órdenes")

## Análisis de Rentabilidad

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS DE RENTABILIDAD")
print("="*80)

total_ganancia = df_orders["Ganancia"].sum()
ganancia_promedio = df_orders["Ganancia"].mean()
margen_promedio = (df_orders["Ganancia"] / df_orders["Ventas"] * 100).mean()
ordenes_rentables = df_orders[df_orders["Ganancia"] > 0]["ID_orden"].nunique()
pct_rentables = (ordenes_rentables / total_ordenes) * 100

print(f"\n💸 GANANCIAS:")
print(f"   Ganancia total: ${total_ganancia:,.2f}")
print(f"   Ganancia promedio por línea: ${ganancia_promedio:,.2f}")
print(f"   Margen promedio: {margen_promedio:.2f}%")
print(f"   Órdenes rentables: {ordenes_rentables:,} ({pct_rentables:.1f}%)")
print(f"   Órdenes con pérdida: {total_ordenes - ordenes_rentables:,} ({100-pct_rentables:.1f}%)")

# Descuentos
ordenes_con_descuento = df_orders[df_orders["Descuento"] > 0]["ID_orden"].nunique()
pct_descuento = (ordenes_con_descuento / total_ordenes) * 100

print(f"\n🎟️ DESCUENTOS:")
print(f"   Órdenes con descuento: {ordenes_con_descuento:,} ({pct_descuento:.1f}%)")

## Top 10 Productos

In [ ]:
print("\n📊 TOP 10 PRODUCTOS (por ventas):")
top_productos = df_orders.groupby("Nombre_producto")["Ventas"].agg(["sum", "count"]).sort_values("sum", ascending=False)
top_productos.columns = ["Ventas_Total", "Cantidad_Vendida"]

for idx, (prod, row) in enumerate(top_productos.head(10).iterrows(), 1):
    pct_ventas = (row["Ventas_Total"] / total_ventas) * 100
    print(f"   {idx}. {prod}: ${row['Ventas_Total']:,.0f} ({pct_ventas:.1f}%)")

## Visualización: Top 10 Productos

In [ ]:
fig = px.bar(
    top_productos.reset_index().head(10),
    x='Nombre_producto',
    y='Ventas_Total',
    title='Top 10 Productos por Ventas',
    labels={'Ventas_Total': 'Ventas ($)', 'Nombre_producto': 'Producto'},
    color='Ventas_Total',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

## Visualización: Top 5 Países

In [ ]:
fig = px.bar(
    top_paises.reset_index().head(5),
    x='País',
    y='Ventas_Total',
    title='Top 5 Países por Ventas',
    labels={'Ventas_Total': 'Ventas ($)', 'País': 'País'},
    color='Ventas_Total',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=500)
fig.show()

## Visualización: Distribución por Mercado

In [ ]:
mercados = df_orders.groupby("Mercado")["Ventas"].sum().reset_index()

fig = px.pie(
    mercados,
    values='Ventas',
    names='Mercado',
    title='Distribución de Ventas por Mercado'
)
fig.show()

## Visualización: Tendencia de Ventas Mensuales

In [ ]:
df_orders['AñoMes'] = df_orders['Fecha_de_la_orden'].dt.to_period('M').astype(str)
ventas_mes = df_orders.groupby('AñoMes')["Ventas"].sum().reset_index()

fig = px.line(
    ventas_mes,
    x='AñoMes',
    y='Ventas',
    title='Tendencia de Ventas Mensuales',
    markers=True,
    labels={'Ventas': 'Ventas ($)', 'AñoMes': 'Año-Mes'}
)
fig.show()

## ✅ Análisis Completado

Para ver más análisis (Segmentación RFM, Análisis Pareto, etc.), revisa el notebook completo en:

📍 **GitHub:** https://github.com/Jextends212/Reto-Aux-Analista

📖 **Documentación:** [Revisar RESUMEN_EJECUTIVO_EDA.md](https://github.com/Jextends212/Reto-Aux-Analista/blob/main/RESUMEN_EJECUTIVO_EDA.md)